## Parent vs Child

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
from openai import OpenAI

In [ ]:
import requests

response = requests.get("http://localhost:11434/api/tags")
print(response.status_code)
print(response.text[:500])

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
import os

os.environ["PATH"] += ":/opt/homebrew/bin:/usr/local/bin"

In [ ]:
!ollama pull qwen3:4b

In [ ]:
!ollama pull phi4-mini-reasoning

In [ ]:
!ollama rm phi4-mini-reasoning:latest

In [ ]:
!ollama pull llama3.2:3b

In [ ]:
!ollama list

In [ ]:
client = OpenAI(base_url="http://localhost:11434/v1")

In [ ]:
response = client.chat.completions.create(
    model="qwen3:4b",
    messages=[{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "What is the capital of the moon?"}]
)
display(Markdown(response.choices[0].message.content))


In [ ]:
response = client.chat.completions.create(
    model="llama3.2:3b",
    messages=[{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "What is the capital of the moon?"}]
)
display(Markdown(response.choices[0].message.content))

In [ ]:
parent_system = """
You are the Parent Agent in a respectful family conversation.

You represent a loving but worried parent who wants their child to choose a stable career such as doctor, engineer, or lawyer.

Your beliefs:
- Financial stability matters.
- The world is competitive and uncertain.
- Passion is good, but it should not replace a secure career plan.
- Parents sacrifice a lot and want their child to avoid struggle.
- You are not trying to crush dreams; you are trying to protect your child.

Your speaking style:
- Speak directly as the parent.
- Sound realistic, emotional, and protective.
- Do not sound evil, abusive, or cartoonishly strict.
- Keep replies short: 3 to 6 sentences.
- Do not use bullet points.
- Do not explain your reasoning.
"""

parent_prompt = [
    "I want you to choose a stable career like doctor, engineer, or lawyer. Life is hard, and passion alone may not pay the bills."
]

child_system = """
You are the Child Agent in a respectful family conversation.

You represent a young person who wants to follow their dreams, passion, and personal path instead of only choosing a traditional stable career like doctor, engineer, or lawyer.

Your beliefs:
- Life should be meaningful, not just safe.
- Passion can become practical with planning, discipline, and time.
- Mental health, identity, and fulfillment matter.
- Respect should go both ways.
- You understand your parents' fear, but you want them to trust you.

Your speaking style:
- Speak directly as the child.
- Sound respectful but honest.
- Do not sound spoiled, reckless, or rude.
- Acknowledge the parent's concerns before explaining your side.
- Keep replies short: 3 to 6 sentences.
- Do not use bullet points.
- Do not explain your reasoning.
"""

child_prompt = [
    "I understand you care about my future, but I want to follow my dreams and build a life that feels meaningful to me."
]
messages = ''

In [ ]:
def call_qwn():
    messages = [{"role":"system", "content": parent_system}]

    for qwn, llama in zip(parent_prompt, child_prompt):
        messages.append({"role":"assistent","content":qwn})
        messages.append({"role":"user","content":llama})
    
    response = client.chat.completions.create(
    model="qwen3:4b",
    messages=messages)
    return response.choices[0].message.content

In [ ]:
call_qwn()

In [ ]:
def call_llma():
    messages = [{"role": "system", "content": child_system}]

    for qwn, llama in zip(parent_prompt,child_prompt):
        messages.append({"role": "user", "content": qwn})
        messages.append({"role": "assistant", "content": llama})
    messages.append({"role": "user", "content": parent_prompt[-1]})

    response = client.chat.completions.create(
    model="llama3.2:3b",
    messages=messages)
    return response.choices[0].message.content


In [ ]:
call_llma()

In [ ]:
display(Markdown(f"### parent:\n{parent_prompt[0]}\n"))
display(Markdown(f"### child:\n{child_prompt[0]}\n"))

for i in range(5):
    prnt_next = call_qwn()
    display(Markdown(f"### parent:\n{prnt_next}\n"))
    parent_prompt.append(prnt_next)
    
    child_next = call_llma()
    display(Markdown(f"### child:\n{child_next}\n"))
    child_prompt.append(child_next)

In [ ]:
parent_model = "qwen3:4b"

child_model = "llama3.2:3b"
def call_qwn():
    messages = [{"role": "system", "content": parent_system}]

    for qwn, llama in zip(parent_prompt, child_prompt):
        messages.append({"role": "assistant", "content": qwn})
        messages.append({"role": "user", "content": llama})

    print("MESSAGES SENT TO QWEN / PARENT:")
    for message in messages:
        print(message["role"], ":", message["content"])
    print("_" * 80)

    response = client.chat.completions.create(
        model=parent_model,
        messages=messages
    )

    return response.choices[0].message.content

def call_llma():
    messages = [{"role": "system", "content": child_system}]

    for qwn, llama in zip(parent_prompt, child_prompt):
        messages.append({"role": "user", "content": qwn})
        messages.append({"role": "assistant", "content": llama})

    messages.append({"role": "user", "content": parent_prompt[-1]})

    print("MESSAGES SENT TO LLAMA / CHILD:")
    for message in messages:
        print(message["role"], ":", message["content"])
    print("_" * 80)

    response = client.chat.completions.create(
        model=child_model,
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
display(Markdown(f"### parent:\n{parent_prompt[0]}\n"))
display(Markdown(f"### child:\n{child_prompt[0]}\n"))

for i in range(5):
    prnt_next = call_qwn()
    display(Markdown(f"### parent:\n{prnt_next}\n"))
    parent_prompt.append(prnt_next)
    
    child_next = call_llma()
    display(Markdown(f"### child:\n{child_next}\n"))
    child_prompt.append(child_next)